<a href="https://colab.research.google.com/github/RajeshworM/Yield_Modelling_Automation/blob/main/NASA_DATA_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================================================
# NASA POWER DAILY DATA DOWNLOAD
# ALL INDIAN DISTRICTS

# =========================================================
# INSTALL PACKAGES
# =========================================================
!pip -q install geopandas requests pyarrow tqdm shapely fiona

# =========================================================
# IMPORTS
# =========================================================
import os
import time
import zipfile
import requests
import pandas as pd
import geopandas as gpd

from tqdm import tqdm

# =========================================================
# OUTPUT DIRECTORIES
# =========================================================
OUTPUT_DIR = "/content/nasa_power_india"

DISTRICT_DIR = os.path.join(OUTPUT_DIR, "district_csvs")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DISTRICT_DIR, exist_ok=True)

# =========================================================
# DOWNLOAD INDIA DISTRICT SHAPEFILE
# =========================================================

print("Downloading India district shapefile...")

shp_url = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbOpen/IND/ADM2/geoBoundaries-IND-ADM2-all.zip"

zip_path = "/content/india_districts.zip"

r = requests.get(shp_url)

with open(zip_path, "wb") as f:
    f.write(r.content)

extract_dir = "/content/india_districts"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Shapefile extracted.")

# =========================================================
# FIND SHAPEFILE
# =========================================================

shp_file = None

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".shp") and "simplified" not in file:
            shp_file = os.path.join(root, file)

print("Using shapefile:", shp_file)

# =========================================================
# LOAD SHAPEFILE
# =========================================================

gdf = gpd.read_file(shp_file)

print("Total districts:", len(gdf))

# =========================================================
# FIX CENTROID WARNING
# =========================================================

# Reproject to metric CRS
gdf_proj = gdf.to_crs(epsg=3857)

# Compute centroids
centroids = gdf_proj.centroid

# Back to WGS84
centroids = gpd.GeoSeries(centroids, crs=3857).to_crs(epsg=4326)

gdf["lon"] = centroids.x
gdf["lat"] = centroids.y

# =========================================================
# DISTRICT NAME COLUMN
# =========================================================

district_col = "shapeName"

if district_col not in gdf.columns:
    district_col = gdf.columns[0]

print("District column:", district_col)

# =========================================================
# NASA POWER PARAMETERS
# VERIFIED DAILY + AG COMMUNITY
# =========================================================

PARAMETERS = [

    # =====================================================
    # SOLAR
    # =====================================================
    "ALLSKY_SFC_SW_DWN",
    "CLRSKY_SFC_SW_DWN",
    "ALLSKY_SFC_SW_DNI",
    "ALLSKY_SFC_SW_DIFF",
    "TOA_SW_DWN",
    "ALLSKY_SFC_PAR_TOT",
    "CLRSKY_SFC_PAR_TOT",

    # =====================================================
    # TEMPERATURE
    # =====================================================
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "T2M_RANGE",
    "T2MDEW",
    "TS",
    "ALLSKY_SFC_LW_DWN",

    # =====================================================
    # HUMIDITY / PRECIP
    # =====================================================
    "RH2M",
    "QV2M",
    "PRECTOTCORR",

    # =====================================================
    # PRESSURE / WIND
    # =====================================================
    "PS",
    "WS2M",
    "WS2M_MAX",
    "WS2M_MIN",
    "WS2M_RANGE",
    "WD2M",

    "WS10M",
    "WS10M_MAX",
    "WS10M_MIN",
    "WS10M_RANGE",
    "WD10M",

    # =====================================================
    # SOIL
    # =====================================================
    "GWETTOP",
    "GWETROOT",
    "GWETPROF",
]

print("Total parameters:", len(PARAMETERS))

# =========================================================
# NASA LIMIT = 20 PARAMETERS MAX
# =========================================================

def chunk_list(lst, size=20):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

PARAMETER_CHUNKS = list(chunk_list(PARAMETERS, 20))

print("Total parameter chunks:", len(PARAMETER_CHUNKS))

# =========================================================
# NASA API SETTINGS
# =========================================================

BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

START_DATE = "20260101"

END_DATE = pd.Timestamp.today().strftime("%Y%m%d")

# =========================================================
# REQUEST FUNCTION
# =========================================================

def fetch_power_data(lat, lon, parameter_chunk):

    params = {
        "parameters": ",".join(parameter_chunk),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }

    try:

        r = requests.get(BASE_URL, params=params, timeout=120)

        if r.status_code != 200:

            print("ERROR:", r.status_code)
            print(r.text[:500])

            return None

        return r.json()

    except Exception as e:

        print("REQUEST ERROR:", e)

        return None

# =========================================================
# PARSE NASA RESPONSE
# =========================================================

def parse_response(js):

    try:

        param_data = js["properties"]["parameter"]

        dfs = []

        for param_name, values in param_data.items():

            temp = pd.DataFrame({
                "date": list(values.keys()),
                param_name: list(values.values())
            })

            dfs.append(temp)

        merged = dfs[0]

        for d in dfs[1:]:

            merged = merged.merge(
                d,
                on="date",
                how="outer"
            )

        return merged

    except Exception as e:

        print("PARSE ERROR:", e)

        return None

# =========================================================
# MAIN DOWNLOAD LOOP
# =========================================================

all_frames = []

for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):

    district = str(row[district_col]).replace("/", "_")

    lat = row["lat"]
    lon = row["lon"]

    out_csv = os.path.join(
        DISTRICT_DIR,
        f"{district}.csv"
    )

    # =====================================================
    # SKIP IF EXISTS
    # =====================================================

    if os.path.exists(out_csv):

        print("Skipping existing:", district)

        continue

    print(f"\nDownloading: {district}")

    district_df = None

    # =====================================================
    # DOWNLOAD PARAMETER CHUNKS
    # =====================================================

    for chunk_idx, chunk in enumerate(PARAMETER_CHUNKS):

        print(
            f"  Chunk {chunk_idx+1}/{len(PARAMETER_CHUNKS)}"
        )

        success = False

        # Retry attempts
        for attempt in range(5):

            try:

                js = fetch_power_data(
                    lat,
                    lon,
                    chunk
                )

                if js is not None:

                    temp_df = parse_response(js)

                    if temp_df is not None:

                        if district_df is None:

                            district_df = temp_df

                        else:

                            district_df = district_df.merge(
                                temp_df,
                                on="date",
                                how="outer"
                            )

                        success = True

                        break

            except Exception as e:

                print(
                    f"Retry {attempt+1} error:",
                    e
                )

            wait_time = (attempt + 1) * 5

            print(f"Waiting {wait_time}s...")

            time.sleep(wait_time)

        # =================================================
        # CHUNK FAILURE
        # =================================================

        if not success:

            print(
                "FAILED CHUNK:",
                district,
                chunk_idx + 1
            )

        # =================================================
        # AVOID NASA RATE LIMIT
        # =================================================

        time.sleep(0.7)

    # =====================================================
    # SAVE DISTRICT FILE
    # =====================================================

    if district_df is not None:

        district_df["district"] = district
        district_df["latitude"] = lat
        district_df["longitude"] = lon

        cols = [
            "district",
            "latitude",
            "longitude",
            "date"
        ] + [
            c for c in district_df.columns
            if c not in [
                "district",
                "latitude",
                "longitude",
                "date"
            ]
        ]

        district_df = district_df[cols]

        district_df.to_csv(
            out_csv,
            index=False
        )

        all_frames.append(district_df)

        print("Saved:", district)

    else:

        print("FAILED DISTRICT:", district)

# =========================================================
# MERGE ALL DISTRICTS
# =========================================================

print("\nCreating merged dataset...")

if len(all_frames) > 0:

    merged = pd.concat(
        all_frames,
        ignore_index=True
    )

    merged["date"] = pd.to_datetime(
        merged["date"]
    )

    # =====================================================
    # SAVE CSV
    # =====================================================

    merged_csv = os.path.join(
        OUTPUT_DIR,
        "india_nasa_power_daily.csv"
    )

    merged.to_csv(
        merged_csv,
        index=False
    )

    # =====================================================
    # SAVE PARQUET
    # =====================================================

    merged_parquet = os.path.join(
        OUTPUT_DIR,
        "india_nasa_power_daily.parquet"
    )

    merged.to_parquet(
        merged_parquet,
        index=False
    )

    print("\nDONE")

    print("CSV:")
    print(merged_csv)

    print("\nPARQUET:")
    print(merged_parquet)

else:

    print("No data downloaded.")

Shapefile extracted.
Using shapefile: /content/india_districts/geoBoundaries-IND-ADM2.shp
Total districts: 735
District column: shapeName
Total parameters: 31
Total parameter chunks: 2


100%|██████████| 735/735 [00:00<00:00, 9129.93it/s]

Skipping existing: Ashoknagar
Skipping existing: Raisen
Skipping existing: Chhindwara
Skipping existing: Betul
Skipping existing: Hoshangabad
Skipping existing: Sehore
Skipping existing: Jabalpur
Skipping existing: Narsimhapur
Skipping existing: Panna
Skipping existing: Ujjain
Skipping existing: Rewa
Skipping existing: Dindori
Skipping existing: Balaghat
Skipping existing: Barwani
Skipping existing: Satna
Skipping existing: Chhatarpur
Skipping existing: Indore
Skipping existing: Ratlam
Skipping existing: Harda
Skipping existing: Sagar
Skipping existing: Neemuch
Skipping existing: Tikamgarh
Skipping existing: Guna
Skipping existing: Dewas
Skipping existing: Mandsaur
Skipping existing: Khargone (West Nimar)
Skipping existing: Sheopur
Skipping existing: Morena
Skipping existing: Bhind
Skipping existing: Jhabua
Skipping existing: Seoni
Skipping existing: Khandwa (East Nimar)
Skipping existing: Umaria
Skipping existing: Gwalior
Skipping existing: Damoh
Skipping existing: Dhar
Skipping exist

In [3]:
from google.colab import files

files.download('/content/nasa_power_india/india_nasa_power_daily.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>